# S3 — Permutation Test: All Metrics (Other)

Extends `S3_permutation_test.ipynb` to compute permutation null distributions
for **remaining metrics** not covered in S3, enabling metric selection / ablation analysis.

**Metrics already in S3 (commented out here):** |Pearson r|, |Spearman ρ|, distance correlation, ew\_bin\_η²

**Phase structure (crash-safe):**

| Phase | Metrics | Count | Speed |
|---|---|---|---|
| Phase 1 | Covariance, slopes, bin stats (vectorised) | 30 | ~30 min |
| Phase 2 | Distance covariance (loop) | 1 | ~4-6 h |
| Phase 3 | KS + Wasserstein distribution tests (loop) | 4 | ~3-6 h |
| Phase 4 | LOWESS (loop, optional) | 5 | ~80 h |
| Assembly | Z-scores + joint test | — | ~2 min |

MINE and GAM are **skipped** (~300+ hours each). LOWESS is optional.

**Input:** `generated_scatterplot_data/cases.csv` + `scatter_points.npz`

**Output:** `generated_scatterplot_data/full/S3_other/permutation_test_all_metrics.parquet`

In [6]:
from __future__ import annotations
import time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import rankdata, ks_2samp, wasserstein_distance

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw): return it

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess
    HAS_LOWESS = True
except ImportError:
    HAS_LOWESS = False

warnings.filterwarnings("ignore")

DATA_DIR  = Path("generated_scatterplot_data")
OUT_DIR   = DATA_DIR / "full" / "S3_other"
OUT_DIR.mkdir(parents=True, exist_ok=True)
N_PERM    = 500
SEED_BASE = 42_000_000

# Metrics where LOWER observed value indicates a relationship
LOWER_IS_SIGNAL = {
    "ew_bin_bw_mean", "ew_bin_bw_early", "ew_bin_bw_mid", "ew_bin_bw_late",
    "ec_bin_bw_mean", "ec_bin_bw_early", "ec_bin_bw_mid", "ec_bin_bw_late",
}

## Load Data

In [7]:
cases_df = pd.read_csv(DATA_DIR / "cases.csv", low_memory=False)
data = np.load(DATA_DIR / "scatter_points.npz")
x_all, y_all = data["x"], data["y"]
n_cases, n_points = x_all.shape
print(f"Loaded {n_cases:,} cases × {n_points} points")

Loaded 114,176 cases × 500 points


## Helper Functions

In [8]:
def _generate_perms(n, n_perm, seed):
    rng = np.random.default_rng(seed)
    return np.array([rng.permutation(n) for _ in range(n_perm)])


def _double_center(a):
    D = squareform(pdist(a.reshape(-1, 1)))
    return D - D.mean(axis=0, keepdims=True) - D.mean(axis=1, keepdims=True) + D.mean()


def _precompute_bins(x, n_bins=10, min_count=5, bin_type="equal_width"):
    try:
        if bin_type == "equal_width":
            bins = np.asarray(pd.cut(x, bins=n_bins, labels=False,
                                     include_lowest=True, duplicates="drop"), dtype=float)
        else:
            bins = np.asarray(pd.qcut(x, q=n_bins, labels=False,
                                      duplicates="drop"), dtype=float)
    except Exception:
        return None
    valid_ids = np.unique(bins[~np.isnan(bins)]).astype(int)
    masks, counts = [], []
    for b in valid_ids:
        m = bins == b
        if m.sum() >= min_count:
            masks.append(m)
            counts.append(m.sum())
    if len(masks) < 2:
        return None
    B_ind = np.array([m.astype(np.float64) for m in masks])
    return B_ind, np.array(counts, dtype=np.float64)


def _z_and_p(obs, null, direction=1):
    """Compute Z-score and p-value. direction=-1 for metrics where lower=signal."""
    med = float(np.median(null))
    iqr = float(np.percentile(null, 75) - np.percentile(null, 25))
    if iqr < 1e-12:
        iqr = float(np.std(null)) * 1.35
    if iqr < 1e-12:
        return 0.0, np.zeros_like(null), med, iqr, 1.0
    if direction == -1:
        z_obs = (med - obs) / iqr
        z_null = (med - null) / iqr
    else:
        z_obs = (obs - med) / iqr
        z_null = (null - med) / iqr
    p = float(np.sum(null >= obs if direction == 1 else null <= obs) + 1) / (len(null) + 1)
    return float(z_obs), z_null, med, iqr, p

## Phase 1 — Vectorised Metrics (~30 min)

30 metrics: covariance (1) + slopes (20) + bin stats (9)

*(abs\_pearson\_r, abs\_spearman\_rho, ew\_bin\_eta2 已在 S3 中计算，此处注释掉)*

In [9]:
def compute_phase1_case(x, y, perms):
    """All vectorisable metrics for one case. Returns {name: (obs, null_1d)}."""
    n = len(x)
    n_perm = len(perms)
    R = {}
    y_perms = y[perms]  # (n_perm, n)

    # ── Correlations ──
    xc = x - x.mean(); yc = y - y.mean()
    sx = np.sqrt((xc**2).sum()); sy = np.sqrt((yc**2).sum())
    if sx > 0 and sy > 0:
        # abs_pearson_r: 已在 S3_permutation_test.ipynb 中计算
        # R["abs_pearson_r"]  = (abs(float((xc*yc).sum()/(sx*sy))),
        #                        np.abs((xc*yc[perms]).sum(1)/(sx*sy)))
        R["abs_covariance"] = (abs(float((xc*yc).sum()/(n-1))),
                               np.abs((xc*yc[perms]).sum(1)/(n-1)))
    else:
        # R["abs_pearson_r"]  = (0., np.zeros(n_perm))
        R["abs_covariance"] = (0., np.zeros(n_perm))

    # abs_spearman_rho: 已在 S3_permutation_test.ipynb 中计算
    # xr = rankdata(x).astype(np.float64); yr = rankdata(y).astype(np.float64)
    # xrc = xr-xr.mean(); yrc = yr-yr.mean()
    # sxr = np.sqrt((xrc**2).sum()); syr = np.sqrt((yrc**2).sum())
    # if sxr > 0 and syr > 0:
    #     R["abs_spearman_rho"] = (abs(float((xrc*yrc).sum()/(sxr*syr))),
    #                               np.abs((xrc*yrc[perms]).sum(1)/(sxr*syr)))
    # else:
    #     R["abs_spearman_rho"] = (0., np.zeros(n_perm))

    # ── Slopes (raw + standardized) ──
    sort_idx = np.argsort(x)
    n1, n2 = n//3, 2*n//3
    seg_slices = {"overall": slice(None), "early": slice(None,n1),
                  "mid": slice(n1,n2), "late": slice(n2,None)}

    for pfx, sxs, syo, syp in [
        ("raw", x[sort_idx], y[sort_idx], y_perms[:, sort_idx]),
        ("std", None, None, None),
    ]:
        if pfx == "std":
            xlo, xhi = x.min(), x.max()
            xn = (x-xlo)/(xhi-xlo) if xhi>xlo else np.full(n, 0.5)
            sxs = xn[sort_idx]
            ylo_p = y_perms.min(1, keepdims=True)
            yhi_p = y_perms.max(1, keepdims=True)
            yr_p  = yhi_p - ylo_p
            syp = np.where(yr_p>0, (y_perms-ylo_p)/yr_p, 0.5)[:, sort_idx]
            ylo_o, yhi_o = y.min(), y.max()
            syo = ((y-ylo_o)/(yhi_o-ylo_o) if yhi_o>ylo_o else np.full(n,0.5))[sort_idx]

        ep_seg_obs, ep_seg_null = [], []
        for sname, slc in seg_slices.items():
            seg_x  = sxs[slc]
            seg_yo = syo[slc]
            seg_yp = syp[:, slc]
            if len(seg_x) < 2:
                R[f"abs_{pfx}_ep_{sname}"] = (0., np.zeros(n_perm))
                R[f"abs_{pfx}_pf_{sname}"] = (0., np.zeros(n_perm))
                continue
            # Endpoint
            dx = seg_x[-1] - seg_x[0]
            if abs(dx) > 0:
                ep_o = abs(float((seg_yo[-1]-seg_yo[0])/dx))
                ep_n = np.abs((seg_yp[:,-1]-seg_yp[:,0])/dx)
            else:
                ep_o = 0.; ep_n = np.zeros(n_perm)
            R[f"abs_{pfx}_ep_{sname}"] = (ep_o, ep_n)
            # Polyfit
            seg_xc = seg_x - seg_x.mean()
            denom_pf = (seg_xc**2).sum()
            if denom_pf > 0 and len(seg_x) >= 3:
                pf_o = abs(float((seg_xc*(seg_yo-seg_yo.mean())).sum()/denom_pf))
                pf_n = np.abs((seg_xc*(seg_yp-seg_yp.mean(1,keepdims=True))).sum(1)/denom_pf)
            else:
                pf_o = 0.; pf_n = np.zeros(n_perm)
            R[f"abs_{pfx}_pf_{sname}"] = (pf_o, pf_n)
            if sname != "overall":
                ep_seg_obs.append(ep_o); ep_seg_null.append(ep_n)

        if ep_seg_null:
            R[f"{pfx}_seg_strength"] = (float(np.mean(ep_seg_obs)),
                                        np.mean(ep_seg_null, axis=0))

    # ── Bin metrics (equal-width + equal-count) ──
    for bt, abbr in [("equal_width","ew"), ("equal_count","ec")]:
        bi = _precompute_bins(x, bin_type=bt)
        y_mean = float(y.mean())
        ss_tot = float(((y-y_mean)**2).sum())

        if bi is None or ss_tot <= 0:
            # ew_bin_eta2: 已在 S3_permutation_test.ipynb 中计算，仅保留 ec
            fallback_names = [f"{abbr}_bin_amp",
                              f"{abbr}_bin_bw_mean", f"{abbr}_bin_bw_early",
                              f"{abbr}_bin_bw_mid", f"{abbr}_bin_bw_late"]
            if abbr == "ec":
                fallback_names.insert(0, f"{abbr}_bin_eta2")
            for nm in fallback_names:
                R[nm] = (0., np.zeros(n_perm))
            continue

        B_ind, bcounts = bi
        nb = len(bcounts)
        bm_obs  = (B_ind @ y) / bcounts
        bm_null = (y_perms @ B_ind.T) / bcounts  # (n_perm, nb)

        # η² — ew_bin_eta2 已在 S3_permutation_test.ipynb 中计算，仅保留 ec
        if abbr == "ec":
            R[f"{abbr}_bin_eta2"] = (
                float((bcounts*(bm_obs-y_mean)**2).sum()/ss_tot),
                (bcounts*(bm_null-y_mean)**2).sum(1)/ss_tot)

        # Amplitude
        R[f"{abbr}_bin_amp"] = (
            float(bm_obs.max()-bm_obs.min()),
            bm_null.max(1)-bm_null.min(1))

        # Buffer widths per bin
        masks_bool = [B_ind[b].astype(bool) for b in range(nb)]
        bw_obs_arr  = np.empty(nb)
        bw_null_arr = np.empty((n_perm, nb))
        for b in range(nb):
            m = masks_bool[b]
            yb_o = y[m]
            bw_obs_arr[b] = np.percentile(yb_o,95) - np.percentile(yb_o,5)
            yb_n = y_perms[:, m]  # (n_perm, n_in_bin)
            bw_null_arr[:,b] = np.percentile(yb_n,95,axis=1) - np.percentile(yb_n,5,axis=1)

        R[f"{abbr}_bin_bw_mean"] = (float(bw_obs_arr.mean()), bw_null_arr.mean(1))
        i1b, i2b = nb//3, 2*nb//3
        for nm, slc in [(f"{abbr}_bin_bw_early", slice(None,max(i1b,1))),
                        (f"{abbr}_bin_bw_mid",   slice(max(i1b,1),max(i2b,i1b+1))),
                        (f"{abbr}_bin_bw_late",  slice(max(i2b,i1b+1),None))]:
            o = float(bw_obs_arr[slc].mean()) if len(bw_obs_arr[slc]) else 0.
            n_ = bw_null_arr[:,slc].mean(1) if bw_null_arr[:,slc].shape[1]>0 else np.zeros(n_perm)
            R[nm] = (o, n_)

    return R

In [10]:
# Run Phase 1
PHASE1_METRICS = None  # populated from first case
phase1_obs_all     = {}  # {name: np.array(n_cases)}
phase1_max_z_null  = np.empty((n_cases, N_PERM), dtype=np.float32)
phase1_summaries   = []  # list of dicts per case

t0 = time.time()
for i in tqdm(range(n_cases), desc="Phase 1"):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)
    R = compute_phase1_case(x, y, perms)

    if PHASE1_METRICS is None:
        PHASE1_METRICS = sorted(R.keys())
        for nm in PHASE1_METRICS:
            phase1_obs_all[nm] = np.empty(n_cases)

    row = {}
    z_nulls = []
    for nm in PHASE1_METRICS:
        obs_val, null_arr = R[nm]
        phase1_obs_all[nm][i] = obs_val
        d = -1 if nm in LOWER_IS_SIGNAL else 1
        z_o, z_n, med, iqr, p = _z_and_p(obs_val, null_arr.astype(np.float64), d)
        z_nulls.append(z_n)
        row[f"{nm}_obs"] = obs_val
        row[f"{nm}_null_med"] = med
        row[f"{nm}_null_iqr"] = iqr
        row[f"z_{nm}"] = z_o
        row[f"p_{nm}"] = p

    phase1_max_z_null[i] = np.stack(z_nulls).max(axis=0).astype(np.float32)
    phase1_summaries.append(row)

elapsed = time.time() - t0
print(f"\nPhase 1 done: {n_cases:,} cases, {len(PHASE1_METRICS)} metrics, {elapsed/60:.1f} min")

ckpt = OUT_DIR / "_allmetrics_phase1.npz"
np.savez_compressed(ckpt, max_z_null=phase1_max_z_null)
print(f"Checkpoint: {ckpt} ({ckpt.stat().st_size/1e6:.0f} MB)")
print(f"Metrics: {PHASE1_METRICS}")

Phase 1: 100%|██████████| 114176/114176 [2:00:21<00:00, 15.81it/s] 



Phase 1 done: 114,176 cases, 30 metrics, 120.4 min
Checkpoint: generated_scatterplot_data/full/S3_other/_allmetrics_phase1.npz (200 MB)
Metrics: ['abs_covariance', 'abs_raw_ep_early', 'abs_raw_ep_late', 'abs_raw_ep_mid', 'abs_raw_ep_overall', 'abs_raw_pf_early', 'abs_raw_pf_late', 'abs_raw_pf_mid', 'abs_raw_pf_overall', 'abs_std_ep_early', 'abs_std_ep_late', 'abs_std_ep_mid', 'abs_std_ep_overall', 'abs_std_pf_early', 'abs_std_pf_late', 'abs_std_pf_mid', 'abs_std_pf_overall', 'ec_bin_amp', 'ec_bin_bw_early', 'ec_bin_bw_late', 'ec_bin_bw_mean', 'ec_bin_bw_mid', 'ec_bin_eta2', 'ew_bin_amp', 'ew_bin_bw_early', 'ew_bin_bw_late', 'ew_bin_bw_mean', 'ew_bin_bw_mid', 'raw_seg_strength', 'std_seg_strength']


## Phase 2 — Distance Covariance (~4-6 h)

*(distance correlation 已在 S3 中计算，此处仅计算 distance covariance)*

In [11]:
phase2_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
phase2_summaries  = []

t0 = time.time()
for i in tqdm(range(n_cases), desc="Phase 2"):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

    A = _double_center(x); B = _double_center(y)

    dcov_obs = np.sqrt(max(float((A*B).mean()), 0))

    dcov_null = np.empty(N_PERM)
    for k in range(N_PERM):
        p = perms[k]
        v = float((A * B[p][:, p]).mean())
        dcov_null[k] = np.sqrt(max(v, 0))

    row = {}
    z_nulls = []
    # dcor: 已在 S3_permutation_test.ipynb 中计算，仅保留 dcov
    for nm, obs, null in [("dcov", float(dcov_obs), dcov_null)]:
        z_o, z_n, med, iqr, pv = _z_and_p(obs, null.astype(np.float64), 1)
        z_nulls.append(z_n)
        row[f"{nm}_obs"]=obs; row[f"{nm}_null_med"]=med
        row[f"{nm}_null_iqr"]=iqr; row[f"z_{nm}"]=z_o; row[f"p_{nm}"]=pv

    phase2_max_z_null[i] = np.stack(z_nulls).max(0).astype(np.float32)
    phase2_summaries.append(row)

    if (i+1) % 5000 == 0:
        el = time.time()-t0; r=(i+1)/el
        print(f"  {i+1:>7,}/{n_cases:,} ({r:.1f}/s, ETA {(n_cases-i-1)/r/3600:.1f}h)")

print(f"\nPhase 2 done: {(time.time()-t0)/3600:.1f} h")
ckpt = OUT_DIR / "_allmetrics_phase2.npz"
np.savez_compressed(ckpt, max_z_null=phase2_max_z_null)
print(f"Checkpoint: {ckpt}")

Phase 2:   4%|▍         | 5000/114176 [41:39<16:39:26,  1.82it/s]

    5,000/114,176 (2.0/s, ETA 15.2h)


Phase 2:   9%|▉         | 10000/114176 [1:18:41<13:59:28,  2.07it/s]

   10,000/114,176 (2.1/s, ETA 13.7h)


Phase 2:  13%|█▎        | 15000/114176 [1:52:16<11:06:45,  2.48it/s]

...

  105,000/114,176 (2.3/s, ETA 1.1h)


Phase 2:  96%|█████████▋| 110000/114176 [13:13:38<29:28,  2.36it/s]  

  110,000/114,176 (2.3/s, ETA 0.5h)


Phase 2: 100%|██████████| 114176/114176 [13:44:50<00:00,  2.31it/s]



Phase 2 done: 13.7 h
Checkpoint: generated_scatterplot_data/full/S3_other/_allmetrics_phase2.npz


## Phase 3 — Distribution Tests (~3-6 h)

KS distance + Wasserstein distance × equal-width / equal-count

In [12]:
phase3_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
phase3_summaries  = []

t0 = time.time()
for i in tqdm(range(n_cases), desc="Phase 3"):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)
    y_perms = y[perms]

    row = {}
    z_nulls = []

    for bt, abbr in [("equal_width","ew"), ("equal_count","ec")]:
        bi = _precompute_bins(x, bin_type=bt)
        if bi is None or len(bi[1]) < 3:
            for nm in [f"{abbr}_dist_ks", f"{abbr}_dist_wass"]:
                row[f"{nm}_obs"]=0.; row[f"{nm}_null_med"]=0.
                row[f"{nm}_null_iqr"]=0.; row[f"z_{nm}"]=0.; row[f"p_{nm}"]=1.
                z_nulls.append(np.zeros(N_PERM))
            continue

        B_ind, bcounts = bi
        nv = len(bcounts)
        low_mask  = B_ind[:max(nv//3,1)].max(0).astype(bool)
        high_mask = B_ind[max(2*nv//3, nv//3+1):].max(0).astype(bool)

        y_lo_o, y_hi_o = y[low_mask], y[high_mask]
        if len(y_lo_o)<2 or len(y_hi_o)<2:
            for nm in [f"{abbr}_dist_ks", f"{abbr}_dist_wass"]:
                row[f"{nm}_obs"]=0.; row[f"{nm}_null_med"]=0.
                row[f"{nm}_null_iqr"]=0.; row[f"z_{nm}"]=0.; row[f"p_{nm}"]=1.
                z_nulls.append(np.zeros(N_PERM))
            continue

        ks_obs  = float(ks_2samp(y_lo_o, y_hi_o).statistic)
        w_obs   = float(wasserstein_distance(y_lo_o, y_hi_o))
        ks_null = np.empty(N_PERM); w_null = np.empty(N_PERM)
        for k in range(N_PERM):
            yl = y_perms[k, low_mask]; yh = y_perms[k, high_mask]
            ks_null[k] = ks_2samp(yl, yh).statistic
            w_null[k]  = wasserstein_distance(yl, yh)

        for nm, obs, null in [(f"{abbr}_dist_ks", ks_obs, ks_null),
                               (f"{abbr}_dist_wass", w_obs, w_null)]:
            z_o, z_n, med, iqr, pv = _z_and_p(obs, null, 1)
            z_nulls.append(z_n)
            row[f"{nm}_obs"]=obs; row[f"{nm}_null_med"]=med
            row[f"{nm}_null_iqr"]=iqr; row[f"z_{nm}"]=z_o; row[f"p_{nm}"]=pv

    phase3_max_z_null[i] = np.stack(z_nulls).max(0).astype(np.float32) if z_nulls else 0.
    phase3_summaries.append(row)

    if (i+1) % 5000 == 0:
        el=time.time()-t0; r=(i+1)/el
        print(f"  {i+1:>7,}/{n_cases:,} ({r:.1f}/s, ETA {(n_cases-i-1)/r/3600:.1f}h)")

print(f"\nPhase 3 done: {(time.time()-t0)/3600:.1f} h")
ckpt = OUT_DIR / "_allmetrics_phase3.npz"
np.savez_compressed(ckpt, max_z_null=phase3_max_z_null)
print(f"Checkpoint: {ckpt}")

Phase 3:   4%|▍         | 5000/114176 [22:33<8:02:21,  3.77it/s] 

    5,000/114,176 (3.7/s, ETA 8.2h)


Phase 3:   9%|▉         | 10000/114176 [45:15<7:45:22,  3.73it/s]

   10,000/114,176 (3.7/s, ETA 7.9h)


Phase 3:  13%|█▎        | 15000/114176 [1:07:45<7:23:22,  3.73it/s]

...

  105,000/114,176 (3.6/s, ETA 0.7h)


Phase 3:  96%|█████████▋| 110000/114176 [8:33:47<19:26,  3.58it/s]

  110,000/114,176 (3.6/s, ETA 0.3h)


Phase 3: 100%|██████████| 114176/114176 [8:53:09<00:00,  3.57it/s]



Phase 3 done: 8.9 h
Checkpoint: generated_scatterplot_data/full/S3_other/_allmetrics_phase3.npz


## Phase 4 — LOWESS (Optional, ~80 h)

⚠️ **Very slow.** Skip this cell unless you have time. Set `RUN_LOWESS = True` to enable.

In [13]:
RUN_LOWESS = False  # ← set True to run (~80 hours)

if RUN_LOWESS and HAS_LOWESS:
    phase4_max_z_null = np.empty((n_cases, N_PERM), dtype=np.float32)
    phase4_summaries  = []

    def _lowess_metrics_single(x, y, frac=0.25):
        if len(x) < 5: return {k: 0. for k in ["res_sd","amp","r2","sign_ch","slope"]}
        order = np.argsort(x); xs, ys = x[order], y[order]
        fitted = sm_lowess(ys, xs, frac=frac, return_sorted=True)
        xf, yf = fitted[:,0], fitted[:,1]
        yp = np.interp(xs, xf, yf)
        r = ys - yp
        ss_res = (r**2).sum(); ss_tot = ((ys-ys.mean())**2).sum()
        # sign changes
        dx = np.diff(xf); dy = np.diff(yf)
        ok = dx != 0; slopes = dy[ok]/dx[ok]
        signs = np.zeros_like(slopes, dtype=int)
        signs[slopes>1e-6]=1; signs[slopes<-1e-6]=-1
        nz = signs[signs!=0]
        sc = float(np.sum(nz[1:]!=nz[:-1])) if len(nz)>=2 else 0.
        return {
            "res_sd": float(np.std(r, ddof=1)) if len(r)>=2 else 0.,
            "amp": float(yf.max()-yf.min()),
            "r2": float(1-ss_res/ss_tot) if ss_tot>0 else 0.,
            "sign_ch": sc,
            "slope": abs(float((yf[-1]-yf[0])/(xf[-1]-xf[0]))) if (xf[-1]-xf[0])!=0 else 0.,
        }

    t0 = time.time()
    for i in tqdm(range(n_cases), desc="Phase 4 (LOWESS)"):
        x = x_all[i].astype(np.float64)
        y = y_all[i].astype(np.float64)
        perms = _generate_perms(len(x), N_PERM, SEED_BASE + i)

        obs_m = _lowess_metrics_single(x, y)
        null_m = {k: np.empty(N_PERM) for k in obs_m}
        for k_perm in range(N_PERM):
            nm = _lowess_metrics_single(x, y[perms[k_perm]])
            for k in nm: null_m[k][k_perm] = nm[k]

        row = {}; z_nulls = []
        for k in obs_m:
            nm = f"lowess_{k}"
            d = -1 if k == "res_sd" else 1
            z_o, z_n, med, iqr, pv = _z_and_p(obs_m[k], null_m[k], d)
            z_nulls.append(z_n)
            row[f"{nm}_obs"]=obs_m[k]; row[f"{nm}_null_med"]=med
            row[f"{nm}_null_iqr"]=iqr; row[f"z_{nm}"]=z_o; row[f"p_{nm}"]=pv

        phase4_max_z_null[i] = np.stack(z_nulls).max(0).astype(np.float32)
        phase4_summaries.append(row)

        if (i+1) % 2000 == 0:
            el=time.time()-t0; r=(i+1)/el
            print(f"  {i+1:>7,}/{n_cases:,} ({r:.1f}/s, ETA {(n_cases-i-1)/r/3600:.1f}h)")

    print(f"\nPhase 4 done: {(time.time()-t0)/3600:.1f} h")
    ckpt = OUT_DIR / "_allmetrics_phase4.npz"
    np.savez_compressed(ckpt, max_z_null=phase4_max_z_null)
else:
    phase4_summaries = None
    phase4_max_z_null = None
    print("Phase 4 skipped (RUN_LOWESS=False)")

Phase 4 skipped (RUN_LOWESS=False)


## Assembly — Joint Test & Classification

In [14]:
# Merge all phase summaries
all_rows = []
for i in range(n_cases):
    row = {"case_id": int(cases_df["case_id"].iloc[i])}
    row.update(phase1_summaries[i])
    row.update(phase2_summaries[i])
    row.update(phase3_summaries[i])
    if phase4_summaries is not None:
        row.update(phase4_summaries[i])
    all_rows.append(row)

# Joint T from max-Z across phases
T_null_joint = np.maximum(phase1_max_z_null, phase2_max_z_null)
T_null_joint = np.maximum(T_null_joint, phase3_max_z_null)
if phase4_max_z_null is not None:
    T_null_joint = np.maximum(T_null_joint, phase4_max_z_null)

# Collect all z_ columns for T_obs
z_cols = [c for c in all_rows[0] if c.startswith("z_")]
for i, row in enumerate(all_rows):
    T_obs = max(row[c] for c in z_cols)
    p_val = float(np.sum(T_null_joint[i] >= T_obs) + 1) / (N_PERM + 1)
    row["T_joint"] = T_obs
    row["p_value"] = p_val

print(f"Assembly done: {len(z_cols)} metrics in joint test")

Assembly done: 35 metrics in joint test


## Save

In [15]:
df = pd.DataFrame(all_rows)
df["classification"] = "uncertain"
df.loc[df["p_value"] <= 0.05, "classification"] = "detectable"
df.loc[df["p_value"] >= 0.10, "classification"] = "not_detectable"

out_path = OUT_DIR / "permutation_test_all_metrics.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved {out_path}  ({len(df):,} rows × {len(df.columns)} cols, {out_path.stat().st_size/1e6:.1f} MB)")
print()
print(df["classification"].value_counts().to_string())
print()
display(df.head(5))

Saved generated_scatterplot_data/full/S3_other/permutation_test_all_metrics.parquet  (114,176 rows × 179 cols, 159.6 MB)

classification
detectable        112714
not_detectable       984
uncertain            478



,case_id,abs_covariance_obs,abs_covariance_null_med,abs_covariance_null_iqr,z_abs_covariance,p_abs_covariance,abs_raw_ep_early_obs,abs_raw_ep_early_null_med,abs_raw_ep_early_null_iqr,z_abs_raw_ep_early,...,z_ec_dist_ks,p_ec_dist_ks,ec_dist_wass_obs,ec_dist_wass_null_med,ec_dist_wass_null_iqr,z_ec_dist_wass,p_ec_dist_wass,T_joint,p_value,classification
0,1,0.007172,0.000774,0.000998,6.410016,0.001996,0.074109,0.256778,0.340140,-0.537043,...,4.988764,0.001996,0.056925,0.012356,0.005647,7.892083,0.001996,9.181659,0.001996,detectable
1,2,0.002955,0.000431,0.000529,4.769383,0.001996,1.635555,0.500387,0.599291,1.894185,...,4.300000,0.003992,0.045662,0.011599,0.005024,6.780447,0.001996,6.780447,0.001996,detectable
2,3,0.002000,0.000428,0.000503,3.124254,0.001996,0.027405,0.188304,0.230738,-0.697321,...,2.652174,0.007984,0.028452,0.011569,0.005322,3.172053,0.005988,4.014150,0.003992,detectable
3,4,0.002833,0.000458,0.000564,4.210549,0.001996,0.473104,0.311020,0.382122,0.424169,...,1.777778,0.037924,0.034123,0.012094,0.005223,4.218068,0.001996,5.083292,0.001996,detectable
4,5,0.004319,0.000671,0.000764,4.773286,0.001996,0.377291,0.281661,0.396356,0.241273,...,3.818182,0.003992,0.051215,0.012333,0.004904,7.928331,0.001996,7.928331,0.001996,detectable


## Metric Ranking & Sanity Checks

In [16]:
check = df.merge(
    cases_df[["case_id","family_id","snr","spread_pattern","x_distribution"]],
    on="case_id",
)

# 1) Null false-positive rate
null_mask = check["family_id"]=="Null"
n_null = int(null_mask.sum())
null_fp = int((check.loc[null_mask,"classification"]=="detectable").sum())
print(f"=== Null FP rate: {null_fp}/{n_null} = {null_fp/n_null:.1%} (target ~5%) ===")
print()

# 2) Per-metric individual detection power
#    For each metric, what fraction of signal cases have individual p < 0.05?
signal = check[check["family_id"]!="Null"]
p_cols = [c for c in df.columns if c.startswith("p_") and c != "p_value"]
metric_power = {}
for pc in p_cols:
    nm = pc[2:]  # strip "p_"
    det_rate = (signal[pc] <= 0.05).mean()
    null_fp_rate = (check.loc[null_mask, pc] <= 0.05).mean() if n_null > 0 else float("nan")
    metric_power[nm] = {"detection_rate": det_rate, "null_fp": null_fp_rate}

power_df = pd.DataFrame(metric_power).T.sort_values("detection_rate", ascending=False)
print("=== Individual metric detection power (signal cases, p<0.05) ===")
for nm, row in power_df.iterrows():
    bar = "█" * int(row["detection_rate"] * 40)
    print(f"  {nm:30s}  det={row['detection_rate']:.1%}  FP={row['null_fp']:.1%}  {bar}")
print()

# 3) Detection rate by SNR (joint test)
signal2 = signal.copy()
signal2["snr_num"] = pd.to_numeric(signal2["snr"], errors="coerce")
det_snr = signal2.groupby("snr_num", dropna=True).apply(
    lambda g: (g["classification"]=="detectable").mean()
).reset_index(name="det_rate")
print("=== Joint test detection rate by SNR ===")
for _, r in det_snr.iterrows():
    bar = "█" * int(r["det_rate"] * 40)
    print(f"  SNR={r['snr_num']:7.2f}  {r['det_rate']:.1%}  {bar}")

=== Null FP rate: 92/128 = 71.9% (target ~5%) ===

=== Individual metric detection power (signal cases, p<0.05) ===
  dcov                            det=98.9%  FP=75.8%  ███████████████████████████████████████
  ec_dist_wass                    det=96.0%  FP=57.0%  ██████████████████████████████████████
  ec_bin_eta2                     det=95.8%  FP=7.0%  ██████████████████████████████████████
  ec_bin_amp                      det=95.7%  FP=10.2%  ██████████████████████████████████████
  ew_dist_wass                    det=95.6%  FP=57.8%  ██████████████████████████████████████
  ec_dist_ks                      det=95.4%  FP=57.0%  ██████████████████████████████████████
  ew_dist_ks                      det=95.4%  FP=58.6%  ██████████████████████████████████████
  ec_bin_bw_mean                  det=92.5%  FP=60.2%  ████████████████████████████████████
  ew_bin_amp                      det=91.2%  FP=8.6%  ████████████████████████████████████
  abs_covariance                  det=90.7%